# Arbus: detektoriaus (YOLOX) apmokymas su tavo pažymėtais kadrais

1. **Runtime → Change runtime type → T4 GPU** (nemokamai).
2. Paleisk langelius iš eilės (Shift+Enter).
3. Kai paprašys – įkelk eksporto `.zip` failą iš `data/traffic/export/`.
4. Pabaigoje parsisiųs `yolox_s_arbus.onnx` – įdėk jį į `arbus-agent/models/`.

Trukmė: ~300 kadrų ir 30 epochų ≈ 20–40 min.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                      # pasirink export ...zip
zname = next(iter(up))
os.makedirs('/content/data', exist_ok=True)
zipfile.ZipFile(zname).extractall('/content/data')
print(os.listdir('/content/data'))

In [ ]:
%cd /content
!git clone -q https://github.com/Megvii-BaseDetection/YOLOX
%cd /content/YOLOX
!pip install -q loguru tabulate pycocotools thop ninja psutil tqdm onnx
!pip install -q --no-deps --no-build-isolation -e .
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth

## Mokymas
`-b 16` – partijos dydis (jei trūksta GPU atminties, sumažink iki 8). Epochų skaičius – `max_epoch` faile `arbus_yolox_exp.py` (30).

In [ ]:
%cd /content/YOLOX
!python tools/train.py -f /content/data/arbus_yolox_exp.py -c yolox_s.pth -d 1 -b 16 --fp16 -o

## Rezultatas
Paskutinėse eilutėse ieškok `Average Precision (AP) @[ IoU=0.50 ]` – kuo arčiau 1, tuo geriau. Tada eksportuojam į ONNX ir parsisiunčiam.

In [ ]:
%cd /content/YOLOX
!python tools/export_onnx.py -f /content/data/arbus_yolox_exp.py -c YOLOX_outputs/arbus_yolox_exp/best_ckpt.pth --output-name yolox_s_arbus.onnx
files.download('yolox_s_arbus.onnx')

## Kas toliau (savo kompiuteryje)
1. Parsisiųstą `yolox_s_arbus.onnx` įdėk į aplanką `arbus-agent/models/`.
2. Admin įrankyje: **Zona ir parametrai → Modelis → yolox_s_arbus → Išsaugoti ir perskaičiuoti**.
3. Palygink **Ataskaitą** su ankstesne. Jei blogiau – grąžink modelį `yolox_s`.